# 07 · Evaluación externa bloqueada y de un solo uso

Este es el único notebook autorizado para abrir el bloque externo. Primero
verifica todos los SHA-256. Si ya existe un resultado completo, lo muestra
y no vuelve a puntuar. Se evalúan únicamente los cinco modelos principales
V6; el test externo no selecciona semillas, hiperparámetros ni redes.


In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "intergate").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Ejecute este notebook dentro del repositorio InterGATE.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from intergate.config import CFG
from intergate.runtime_setup import (
    assert_v6_protocol, assert_transport_protocol, prepare_runtime,
)

assert_v6_protocol(CFG)
assert_transport_protocol(CFG)
ARTIFACTS = Path(CFG.ARTIFACTS_ROOT)
RUN_ROOT = ARTIFACTS / "INTERGATE_V6"
RESULTS = PROJECT_ROOT / "results_v6"
RESULTS.mkdir(parents=True, exist_ok=True)


In [ ]:
import hashlib
from intergate.ablation import predict_bundle_on_test
from intergate.losses import compute_metrics_full

LOCK = ARTIFACTS / "protocol_lock_v6.json"
OUT = RESULTS / "07_external_locked"
COMPLETE = OUT / "external_evaluation_COMPLETE.json"
if not LOCK.is_file():
    raise FileNotFoundError("Ejecute primero el notebook 06.")
lock = json.loads(LOCK.read_text(encoding="utf-8"))

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

changed = []
for path_text, expected in lock["files"].items():
    path = Path(path_text)
    if not path.is_file() or sha256(path) != expected["sha256"]:
        changed.append(path_text)
if changed:
    raise RuntimeError("El protocolo cambió después del lock:\n- " + "\n- ".join(changed))
def notebook_source_sha256(path):
    notebook = json.loads(Path(path).read_text(encoding="utf-8"))
    source_only = []
    for cell in notebook.get("cells", []):
        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)
        source_only.append({
            "cell_type": cell.get("cell_type"), "source": str(source),
        })
    encoded = json.dumps(
        source_only, sort_keys=True, ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()
changed_notebooks = [
    path for path, expected in lock["extra"][
        "notebook_source_sha256_ignores_outputs"
    ].items()
    if not Path(path).is_file() or notebook_source_sha256(path) != expected
]
if changed_notebooks:
    raise RuntimeError(
        "Cambió el código de un notebook después del lock:\n- "
        + "\n- ".join(changed_notebooks)
    )
if COMPLETE.is_file():
    print("El externo ya fue evaluado; se muestran resultados guardados.")
    display(pd.read_csv(OUT / "external_metrics_by_seed.csv"))
    raise SystemExit("Evaluación de un solo uso ya completada.")


In [ ]:
# Esta es la primera línea del flujo que abre expresión y etiquetas externas.
runtime = prepare_runtime(CFG, include_external=True)
assert runtime["external_opened"] is True
if len(runtime["test_idx"]) == 0:
    raise RuntimeError("El bloque externo no se cargó.")

OUT.mkdir(parents=True, exist_ok=True)
metric_rows, prediction_rows = [], []
for seed in CFG.STABILITY_SEEDS:
    seed = int(seed)
    bundle_dir = RUN_ROOT / f"seed_{seed}"
    prediction = predict_bundle_on_test(str(bundle_dir), device=CFG.DEVICE)
    metrics = compute_metrics_full(
        prediction["y_true"], prediction["proba"], runtime["label_map"]
    )
    metric_rows.append({"seed": seed, "scope": "all_external", **metrics})
    for cohort_name in sorted(np.unique(prediction["cohort"])):
        mask = prediction["cohort"] == cohort_name
        cohort_metrics = compute_metrics_full(
            prediction["y_true"][mask], prediction["proba"][mask], runtime["label_map"]
        )
        metric_rows.append({
            "seed": seed, "scope": f"cohort::{cohort_name}", **cohort_metrics,
        })
    for row_id, (sample_index, truth, cohort_name, probabilities) in enumerate(zip(
        prediction["test_idx"], prediction["y_true"],
        prediction["cohort"], prediction["proba"],
    )):
        row = {
            "seed": seed, "test_index": int(sample_index),
            "cohort": str(cohort_name), "y_true": int(truth),
            "y_pred": int(np.argmax(probabilities)),
        }
        row.update({
            f"p_{class_name}": float(probabilities[class_id])
            for class_id, class_name in enumerate(runtime["classes"])
        })
        prediction_rows.append(row)

external_metrics = pd.DataFrame(metric_rows)
external_predictions = pd.DataFrame(prediction_rows)
external_metrics.to_csv(OUT / "external_metrics_by_seed.csv", index=False)
external_predictions.to_csv(
    OUT / "external_predictions.csv.gz", index=False, compression="gzip"
)
display(external_metrics[external_metrics.scope.eq("all_external")])


In [ ]:
completion = {
    "status": "complete",
    "protocol_lock_sha256": sha256(LOCK),
    "protocol_version": CFG.TRANSPORT_PROTOCOL_VERSION,
    "evaluated_artifact": "INTERGATE_V6",
    "seeds": list(map(int, CFG.STABILITY_SEEDS)),
    "external_cohorts": list(CFG.TEST_COHORTS),
    "n_external_samples": int(len(runtime["test_idx"])),
    "model_or_hyperparameter_selected_on_external": False,
    "external_opened_once": True,
}
COMPLETE.write_text(
    json.dumps(completion, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Evaluación externa bloqueada completa:", OUT)
